In [1]:
import pandas as pd
import numpy as np

In [2]:
simple_prompt_qwen_df = pd.read_csv("/home/ia368/projetos/imageclef2026-rag/artifacts/results/simple_prompt_qwen3_VL_4B_20260422_192239/simple_prompt_qwen_imageclef2026-rag_20260422_192239.csv")
simple_prompt_qwen_df.head()

,ID,Caption
0,ImageCLEFmedical_Caption_2026_valid_0,Brain MRI showing normal anatomical structures...
1,ImageCLEFmedical_Caption_2026_valid_1,Brain MRI showing symmetrical cerebral hemisph...
2,ImageCLEFmedical_Caption_2026_valid_2,Spinal cord MRI showing disc herniation at L2 ...
3,ImageCLEFmedical_Caption_2026_valid_3,Hip replacement with visible implant and arrow...
4,ImageCLEFmedical_Caption_2026_valid_4,Pelvic X-ray showing bilateral hip replacement...


In [3]:
df_ref = pd.read_csv("/home/ia368/projetos/imageclef2026-rag/artifacts/results/rag_simple_prompt_20260310_113904/simple_prompt_config_imageclef2026-rag_simple_prompt_run_20260310_113904.csv")
df_ref.shape

(19240, 2)

In [4]:
simple_prompt_qwen_df.shape

(19238, 2)

In [6]:
images_missing = set(df_ref["ID"]) - set(simple_prompt_qwen_df["ID"])
print(f"Number of missing images: {len(images_missing)}")
print("Missing image IDs: ", images_missing)

Number of missing images: 2
Missing image IDs:  {'ImageCLEFmedical_Caption_2026_valid_5803', 'ImageCLEFmedical_Caption_2026_valid_5804'}


In [7]:
# there are any duplicated IDs in the new df?
duplicated_ids = simple_prompt_qwen_df[simple_prompt_qwen_df.duplicated(subset=["ID"], keep=False)]
print(f"Number of duplicated IDs: {duplicated_ids.shape[0]}")
if not duplicated_ids.empty:
    print("Duplicated IDs: ", duplicated_ids["ID"].tolist())

Number of duplicated IDs: 0


In [8]:
len(simple_prompt_qwen_df["ID"].unique())

19238

## rodar para IDs faltantes

In [9]:
# ids faltantes = images_missing
from anyio import Path
import json

base = Path("/home/ia368/projetos/imageclef2026-rag")

In [10]:
print("IDs selecionados:", len(images_missing))
print(images_missing)

dataset_full = base / "artifacts/datasets/imageclef2026_valid_dataset.json"

with open(dataset_full, "r", encoding="utf-8") as f:
    data = json.load(f)

subset = [x for x in data if x["image_id"] in set(images_missing)]
found_ids = {x["image_id"] for x in subset}
not_found = [x for x in images_missing if x not in found_ids]

print("Encontrados no dataset:", len(subset))
print("Nao encontrados:", not_found)

subset_path = base / "artifacts/results/simple_prompt_qwen3_VL_4B_20260422_192239/imageclef2026_valid_subset_2_ids.json"
with open(subset_path, "w", encoding="utf-8") as f:
    json.dump(subset, f, ensure_ascii=False, indent=2)

print("Subset salvo em:", subset_path)

IDs selecionados: 2
{'ImageCLEFmedical_Caption_2026_valid_5803', 'ImageCLEFmedical_Caption_2026_valid_5804'}
Encontrados no dataset: 2
Nao encontrados: []
Subset salvo em: /home/ia368/projetos/imageclef2026-rag/artifacts/results/simple_prompt_qwen3_VL_4B_20260422_192239/imageclef2026_valid_subset_2_ids.json


In [11]:
import yaml
from copy import deepcopy

config_src = base / "configs/runs/simple_prompt_qwen.yaml"
config_tmp = base / "artifacts/results/simple_prompt_qwen3_VL_4B_20260422_192239/simple_prompt_qwen_2_ids.yaml"

with open(config_src, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

cfg2 = deepcopy(cfg)
cfg2["dataset"] = str(subset_path)

with open(config_tmp, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg2, f, sort_keys=False, allow_unicode=True)

print("Config temporaria salva em:", config_tmp)

Config temporaria salva em: /home/ia368/projetos/imageclef2026-rag/artifacts/results/simple_prompt_qwen3_VL_4B_20260422_192239/simple_prompt_qwen_2_ids.yaml


In [12]:
csv_main = base / "/home/ia368/projetos/imageclef2026-rag/artifacts/results/simple_prompt_qwen3_VL_4B_20260422_192239/simple_prompt_qwen_imageclef2026-rag_20260422_192239.csv"
cmd = f'python {base/"run_med_gemma.py"} --config {config_tmp} --resume_csv {csv_main}'
print(cmd)

python /home/ia368/projetos/imageclef2026-rag/run_med_gemma.py --config /home/ia368/projetos/imageclef2026-rag/artifacts/results/simple_prompt_qwen3_VL_4B_20260422_192239/simple_prompt_qwen_2_ids.yaml --resume_csv /home/ia368/projetos/imageclef2026-rag/artifacts/results/simple_prompt_qwen3_VL_4B_20260422_192239/simple_prompt_qwen_imageclef2026-rag_20260422_192239.csv


## alterar ordem os IDs no arquivo csv final

In [13]:
simple_prompt_qwen_df = pd.read_csv("/home/ia368/projetos/imageclef2026-rag/artifacts/results/simple_prompt_qwen3_VL_4B_20260422_192239/simple_prompt_qwen_imageclef2026-rag_20260422_192239.csv")
simple_prompt_qwen_df.head()

,ID,Caption
0,ImageCLEFmedical_Caption_2026_valid_0,Brain MRI showing normal anatomical structures...
1,ImageCLEFmedical_Caption_2026_valid_1,Brain MRI showing symmetrical cerebral hemisph...
2,ImageCLEFmedical_Caption_2026_valid_2,Spinal cord MRI showing disc herniation at L2 ...
3,ImageCLEFmedical_Caption_2026_valid_3,Hip replacement with visible implant and arrow...
4,ImageCLEFmedical_Caption_2026_valid_4,Pelvic X-ray showing bilateral hip replacement...


In [15]:
simple_prompt_qwen_df['id_num'] = (
    simple_prompt_qwen_df['ID'].str.split('_').str[-1].astype(int)
)
simple_prompt_qwen_df.head()

,ID,Caption,id_num
0,ImageCLEFmedical_Caption_2026_valid_0,Brain MRI showing normal anatomical structures...,0
1,ImageCLEFmedical_Caption_2026_valid_1,Brain MRI showing symmetrical cerebral hemisph...,1
2,ImageCLEFmedical_Caption_2026_valid_2,Spinal cord MRI showing disc herniation at L2 ...,2
3,ImageCLEFmedical_Caption_2026_valid_3,Hip replacement with visible implant and arrow...,3
4,ImageCLEFmedical_Caption_2026_valid_4,Pelvic X-ray showing bilateral hip replacement...,4


In [16]:
simple_prompt_qwen_df = simple_prompt_qwen_df.sort_values(by="id_num").reset_index()
simple_prompt_qwen_df.head()

,index,ID,Caption,id_num
0,0,ImageCLEFmedical_Caption_2026_valid_0,Brain MRI showing normal anatomical structures...,0
1,1,ImageCLEFmedical_Caption_2026_valid_1,Brain MRI showing symmetrical cerebral hemisph...,1
2,2,ImageCLEFmedical_Caption_2026_valid_2,Spinal cord MRI showing disc herniation at L2 ...,2
3,3,ImageCLEFmedical_Caption_2026_valid_3,Hip replacement with visible implant and arrow...,3
4,4,ImageCLEFmedical_Caption_2026_valid_4,Pelvic X-ray showing bilateral hip replacement...,4


In [17]:
simple_prompt_qwen_df.tail(10)

,index,ID,Caption,id_num
19230,19228,ImageCLEFmedical_Caption_2026_valid_19243,Corneal thickness measurement of 0.23 mm indic...,19243
19231,19229,ImageCLEFmedical_Caption_2026_valid_19244,Optical coherence tomography image showing sub...,19244
19232,19230,ImageCLEFmedical_Caption_2026_valid_19245,Macular Swelling indicated by arrow in OCT scan,19245
19233,19231,ImageCLEFmedical_Caption_2026_valid_19246,Ultrasound image showing a structural anomaly ...,19246
19234,19232,ImageCLEFmedical_Caption_2026_valid_19247,Outer retinal tubulation indicated by arrows i...,19247
19235,19233,ImageCLEFmedical_Caption_2026_valid_19248,Retinal layer disruption with intraretinal flu...,19248
19236,19234,ImageCLEFmedical_Caption_2026_valid_19249,Spectral domain optical coherence tomography i...,19249
19237,19235,ImageCLEFmedical_Caption_2026_valid_19250,Ultrasound image showing a localized hyperecho...,19250
19238,19236,ImageCLEFmedical_Caption_2026_valid_19251,Scleral thickness measurement with ultrasound ...,19251
19239,19237,ImageCLEFmedical_Caption_2026_valid_19252,Scleral buckle with arrow indicating focal are...,19252


In [18]:
simple_prompt_qwen_df.shape

(19240, 4)

In [19]:
simple_prompt_qwen_df.drop(columns=["index", "id_num"], inplace=True)
simple_prompt_qwen_df.head()

,ID,Caption
0,ImageCLEFmedical_Caption_2026_valid_0,Brain MRI showing normal anatomical structures...
1,ImageCLEFmedical_Caption_2026_valid_1,Brain MRI showing symmetrical cerebral hemisph...
2,ImageCLEFmedical_Caption_2026_valid_2,Spinal cord MRI showing disc herniation at L2 ...
3,ImageCLEFmedical_Caption_2026_valid_3,Hip replacement with visible implant and arrow...
4,ImageCLEFmedical_Caption_2026_valid_4,Pelvic X-ray showing bilateral hip replacement...


In [20]:
simple_prompt_qwen_df.to_csv("/home/ia368/projetos/imageclef2026-rag/artifacts/results/simple_prompt_qwen3_VL_4B_20260422_192239/simple_prompt_qwen3_VL_4B_20260422_192239_sorted.csv", index=False)